# Disrupt Quant — getting started

This notebook covers loading observations and running the public example. Research hypotheses and validation choices are yours. Read the challenge rules before interpreting results.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from starter.backtester import load_data, simulate
from starter.metrics import summarize
from strategy import generate_positions
data = load_data(root / 'data' / 'development.csv')
data.head()

In [ ]:
print(data.shape)
print(data.groupby('asset_id').size())
data[['close', 'volume', 'spread_bps']].describe()

In [ ]:
prices = data.pivot(index='date', columns='asset_id', values='close')
(prices[['A01', 'A02', 'A03']] / prices[['A01', 'A02', 'A03']].iloc[0]).plot(figsize=(10, 4), title='Illustrative normalized prices')
plt.ylabel('Initial price = 1')
plt.show()

In [ ]:
daily, attribution = simulate(data, generate_positions)
summarize(daily)

## Next steps

Write down a hypothesis before trying multiple variants. Use internal chronological splits during development. The separate public validation file is deliberately not loaded here; run validation explicitly only when you are ready. The example demonstrates the interface and is not a preferred research approach.

In [20]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("../data/development.csv", parse_dates=["date"])
data = data.sort_values(by=["asset_id", "date"])

data["future_close"] = data.groupby("asset_id")["close"].shift(-5)

data["forward_return_5d"] = data["future_close"] / data["close"] - 1

sample = data[data.asset_id == "A01"][["date", "close", "future_close", "forward_return_5d"]].head(15)
#print(data[data.asset_id == "A01"][["date", "close", "future_close", "forward_return_5d"]].tail(10))

#print(sample)

#print(data.shape)
#print(data.head())

paired = data[["date", "asset_id", "return_20d", "forward_return_5d"]]
paired = paired.dropna(subset=["date", "asset_id", "return_20d", "forward_return_5d"])

# print(data.shape)
# print(paired.shape)

#plt.scatter(paired["return_20d"], paired["forward_return_5d"], alpha=0.1, s=5)
#plt.xlabel("return_20d (past)")
#plt.ylabel("forward_return_5d (future)")
#plt.show()

paired["bucket"] = pd.qcut(paired["return_20d"], q=5, labels=False)
bucket_means = paired.groupby("bucket")["forward_return_5d"].mean()
#print(bucket_means)

data["relative_return_1d"] = data["return_1d"] - data["sector_return"]

paired2 = data[["date", "asset_id", "relative_return_1d", "forward_return_5d"]]
paired2 = paired2.dropna(subset=["relative_return_1d", "forward_return_5d"])
# print(data.shape, paired2.shape)

paired2["bucket"] = pd.qcut(paired2["relative_return_1d"], q=5, labels=False)
# print(paired2.groupby("bucket")["forward_return_5d"].mean())

sample_date = data["date"].iloc[100]
latest = data.loc[data.date == sample_date]

inv_vol = 1 / latest["realized_vol_20d"]
weights = inv_vol / inv_vol.sum()

check = latest[["asset_id", "realized_vol_20d"]].copy()
check["weight"] = weights.values
print(check.sort_values("realized_vol_20d"))
print(weights.sum())

     asset_id  realized_vol_20d    weight
2414      A15          0.010771  0.053770
2412      A13          0.011216  0.051637
2403      A04          0.011255  0.051460
2419      A20          0.011274  0.051373
2406      A07          0.011508  0.050326
2417      A18          0.011606  0.049904
2400      A01          0.011712  0.049451
2408      A09          0.011843  0.048906
2409      A10          0.011999  0.048269
2404      A05          0.012294  0.047111
2422      A23          0.013677  0.042345
2416      A17          0.013941  0.041544
2407      A08          0.013975  0.041445
2405      A06          0.014091  0.041103
2401      A02          0.014197  0.040796
2410      A11          0.014575  0.039737
2415      A16          0.015561  0.037219
2413      A14          0.016545  0.035005
2420      A21          0.017033  0.034002
2421      A22          0.017774  0.032586
2402      A03          0.018249  0.031738
2423      A24          0.018910  0.030627
2418      A19          0.021623  0